In [78]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [79]:
main_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))
main_df['canonical_casenum'] = main_df['title'].apply(lambda x: parse_casenum(x)['canonical_casenum'])
main_df = main_df.sort_values(by=['canonical_casenum', 'date'], ascending=True).reset_index(drop=True)
main_df['times_appeared'] = main_df.groupby('canonical_casenum').cumcount() + 1

In [80]:
df = []
for idx, row in main_df.iterrows():
    if idx==0:
        continue
    prev_row = main_df.iloc[idx-1]
    if prev_row['canonical_casenum'] != row['canonical_casenum']:
        continue
    if prev_row['project_result'] != "DELAYED":
        continue
    if row['project_result'] == "APPLICATION WITHDRAWN":
        continue
    days_between = (pd.to_datetime(row['date']) - pd.to_datetime(prev_row['date'])).days
    prev_members_present = set(list(prev_row['ayes']) + list(prev_row['noes']) + list(prev_row['abstentions']) + list(prev_row['recusals']))
    members_present = set(list(row['ayes']) + list(row['noes']) + list(row['abstentions']) + list(row['recusals']))
    if len(prev_members_present) == 0:
        frac_new = 0
    else:
        frac_new = len(prev_members_present - members_present) / len(prev_members_present)
    new_row = row.copy()
    new_row['days_between'] = days_between
    new_row['frac_new'] = frac_new
    df.append(new_row)

df = pd.DataFrame(df).reset_index(drop=True)
df['outcome'] = 0
df.loc[df['project_result'].isin(["DELAYED", "DENIED"]), 'outcome'] = 1

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "rehearing_delays.parquet"))

In [81]:
pd.crosstab(df['outcome'], df['project_result'])

project_result,APPROVED,APPROVED WITH MAJOR CONDITIONS OR MODIFICATIONS,APPROVED WITH MINOR CONDITIONS OR MODIFICATIONS,DELAYED,DENIED
outcome,,,,,
0,24,9,38,0,0
1,0,0,0,38,3
